In [1]:
import subprocess
subprocess.run(['pip', 'install', 'sqlalchemy', 'ipython-sql'], capture_output=True)

print("Libraries ready ✓")

Libraries ready ✓


In [2]:
import pandas as pd
import sqlite3
from sqlalchemy import create_engine, text

# Create local SQLite database (creates the file if it doesn't exist)
engine = create_engine('sqlite:///olist_database.db')
conn = sqlite3.connect('olist_database.db')
cursor = conn.cursor()

print("Connected to olist_database.db ✓")

Connected to olist_database.db ✓


In [3]:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS customers (
        customer_id             TEXT        PRIMARY KEY,
        customer_unique_id      TEXT        NOT NULL,
        customer_zip_code_prefix INTEGER    NOT NULL,
        customer_city           TEXT        NOT NULL,
        customer_state          TEXT        NOT NULL
    )
''')
conn.commit()
print("customers table created ✓")

customers table created ✓


In [4]:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS orders (
        order_id                        TEXT    PRIMARY KEY,
        customer_id                     TEXT    NOT NULL,
        order_status                    TEXT    NOT NULL,
        order_purchase_timestamp        TEXT,
        order_approved_at               TEXT,
        order_delivered_carrier_date    TEXT,
        order_delivered_customer_date   TEXT,
        order_estimated_delivery_date   TEXT,
        FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
    )
''')
conn.commit()
print("orders table created ✓")

orders table created ✓


In [5]:
cursor.execute('''
    CREATE TABLE IF NOT EXISTS order_items (
        order_id              TEXT      NOT NULL,
        order_item_id         INTEGER   NOT NULL,
        product_id            TEXT      NOT NULL,
        seller_id             TEXT      NOT NULL,
        shipping_limit_date   TEXT,
        price                 REAL      NOT NULL,
        freight_value         REAL      NOT NULL,
        PRIMARY KEY (order_id, order_item_id),
        FOREIGN KEY (order_id) REFERENCES orders(order_id)
    )
''')
conn.commit()
print("order_items table created ✓")

order_items table created ✓


In [6]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
print("Tables in database:")
for t in tables:
    print(" -", t[0])

Tables in database:
 - customers
 - orders
 - order_items


In [7]:
customers_df   = pd.read_csv('customers_clean.csv')
orders_df      = pd.read_csv('orders_clean.csv')
order_items_df = pd.read_csv('order_items_clean.csv')

print("customers rows :", len(customers_df))
print("orders rows    :", len(orders_df))
print("order_items rows:", len(order_items_df))

customers rows : 99441
orders rows    : 99441
order_items rows: 112650


In [8]:
customers_df.to_sql(
    name='customers',
    con=engine,
    if_exists='append',
    index=False
)
print(f"Imported {len(customers_df)} rows into customers ✓")

Imported 99441 rows into customers ✓


In [9]:
orders_df.to_sql(
    name='orders',
    con=engine,
    if_exists='append',
    index=False
)
print(f"Imported {len(orders_df)} rows into orders ✓")

Imported 99441 rows into orders ✓


In [10]:
order_items_df.to_sql(
    name='order_items',
    con=engine,
    if_exists='append',
    index=False
)
print(f"Imported {len(order_items_df)} rows into order_items ✓")

Imported 112650 rows into order_items ✓
